In [15]:
import random
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import ipywidgets as widgets

In [2]:
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'PingFang SC', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

In [16]:
mpl.rcParams['animation.embed_limit'] = 200.0

In [3]:
# --- 1. 游戏配置 ---
TRACK_LENGTH = 32
TRAPS = {
    4: 'Propel', 
    12: 'Propel', 
    17: 'Propel', 
    24: 'Propel',
    11: 'Block', 
    29: 'Block',
    7: 'BlackHole', 
    21: 'BlackHole'
}

COLORS = {
    '陆赫斯': '#E64B35',   # 红
    '西格莉卡': '#3C5488', # 深蓝
    '达尼娅': '#4DBBD5',   # 亮蓝
    '绯雪': '#8491B4',     # 灰紫
    '卡提希娅': '#F39B7F',     # 橙粉
    '菲比': '#00A087',     # 蓝绿
    'Boss': '#2C2C2C',    # 深灰
    '千咲': '#E18727', 
    '莫宁': '#20854E', 
    '琳奈': '#7876B1',
    '爱弥斯': '#6F99AD', 
    '守岸人': '#BC3C29', 
    '珂莱塔': '#EE4C97'
}

In [4]:
# --- 2. 团子基类 ---
class Tuanzi:
    def __init__(self, name):
        self.name = name
        self.color = COLORS[name]
        self.abs_dist = 0
        self.is_boss = (name == 'Boss')
        self.sigelika_debuff = 0
        self.action_log = ""

    @property
    def pos(self):
        if self.abs_dist >= 0: return (self.abs_dist % TRACK_LENGTH) + 1
        return (TRACK_LENGTH - (abs(self.abs_dist) % TRACK_LENGTH)) % TRACK_LENGTH + 1

    def roll(self): 
        # 用于决定顺序的掷骰 (1-3)
        return random.randint(1, 6) if self.is_boss else random.randint(1, 3)

    def movement_roll(self):
        # 实际移动时的掷骰
        return self.roll()

    def on_round_start(self, game): pass
    def check_encounter(self, game): pass
    def modify_steps(self, game, roll): return max(1, roll - self.sigelika_debuff)
    def on_trap_trigger(self, game, trap_type, default_step): return default_step
    def on_turn_end(self, game): pass

class Luhesi(Tuanzi):
    def on_trap_trigger(self, game, trap_type, default_step):
        if trap_type == 'Propel': return default_step + 3
        if trap_type == 'Block': return default_step - 1
        return default_step

class Sigelika(Tuanzi):
    def on_round_start(self, game):
        if game.round_num <= 1: return
        ranks = game.get_rankings()
        my_idx = ranks.index(self)
        targets = ranks[max(0, my_idx-2):my_idx]
        for t in targets: t.sigelika_debuff = 1
        if targets: self.action_log = f"标记减速了 {','.join([t.name for t in targets])}"

class Dania(Tuanzi):
    def __init__(self, name):
        super().__init__(name)
        self.last_roll = 0
    def modify_steps(self, game, roll):
        base = super().modify_steps(game, roll)
        if roll == self.last_roll:
            self.action_log = "连续相同点数，额外+2格"
            base += 2
        self.last_roll = roll
        return base

class Feixue(Tuanzi):
    def __init__(self, name):
        super().__init__(name)
        self.met_boss = False
    def check_encounter(self, game):
        if not self.met_boss and game.round_num >= 3 and self.pos == game.boss.pos:
            self.met_boss = True
            self.action_log = "与Boss同格相遇，被动解锁永久+1！"
    def modify_steps(self, game, roll):
        base = super().modify_steps(game, roll)
        return base + 1 if self.met_boss else base

class Kati(Tuanzi):
    def __init__(self, name):
        super().__init__(name)
        self.skill_unlocked = False
    def on_turn_end(self, game):
        if not self.skill_unlocked and game.get_rankings()[-1] == self:
            self.skill_unlocked = True
            self.action_log = "处于最后一名，60%冲刺被动解锁！"
    def modify_steps(self, game, roll):
        base = super().modify_steps(game, roll)
        if self.skill_unlocked and random.random() < 0.6:
            self.action_log = "触发60%概率，额外+2格"
            return base + 2
        return base

class Feibi(Tuanzi):
    def modify_steps(self, game, roll):
        base = super().modify_steps(game, roll)
        if random.random() < 0.5:
            self.action_log = "触发50%概率，额外+1格"
            return base + 1
        return base

class Qianxiao(Tuanzi):
    def modify_steps(self, game, roll):
        base = super().modify_steps(game, roll)
        if base <= 0: return base
        min_roll = min(game.current_movement_rolls.values())
        if roll == min_roll:
            self.action_log = "掷出本轮最小点数，额外+2格"
            base += 2
        return base

class Moning(Tuanzi):
    def __init__(self, name):
        super().__init__(name)
        self.seq = [3, 2, 1]
        self.seq_idx = 0
    def movement_roll(self):
        res = self.seq[self.seq_idx]
        self.seq_idx = (self.seq_idx + 1) % 3
        return res

class Linnai(Tuanzi):
    def modify_steps(self, game, roll):
        r = random.random()
        if r < 0.20:
            self.action_log = "触发20%负面效果，本轮无法移动"
            return 0
        elif r < 0.80:
            self.action_log = "触发60%增益被动，按照掷骰双倍移动！"
            return max(1, roll * 2 - self.sigelika_debuff)
        return super().modify_steps(game, roll)

class Aimisi(Tuanzi):
    def __init__(self, name):
        super().__init__(name)
        self.skill_used = False
    def on_turn_end(self, game):
        if not self.skill_used and self.abs_dist >= 16:
            ahead = [t for t in game.tuanzis if not t.is_boss and t.abs_dist > self.abs_dist]
            if ahead:
                self.skill_used = True
                nearest_dist = min(t.abs_dist for t in ahead)
                offset = nearest_dist - self.abs_dist
                self.action_log = f"触发空间跃迁！传送至前方最近堆叠顶端"
                game.move_stack(self, offset)
                game.trigger_encounters()
                game.record_state(f"{self.name} 传送跃迁了 {offset} 格", self)

class Shouanren(Tuanzi):
    def movement_roll(self):
        # 决定顺序使用基类的随机1-3，仅在移动时固定为2/3
        return random.choice([2, 3])

class Kelaita(Tuanzi):
    def modify_steps(self, game, roll):
        if random.random() < 0.28:
            self.action_log = "触发28%概率，掷骰结果双倍移动！"
            return max(1, roll * 2 - self.sigelika_debuff)
        return super().modify_steps(game, roll)

In [5]:
# 【重要：角色注册表】
AVAILABLE_TUANZIS = {
    '陆赫斯': Luhesi, 
    '西格莉卡': Sigelika, 
    '达尼娅': Dania,
    '绯雪': Feixue, 
    '卡提希娅': Kati, 
    '菲比': Feibi,
    '千咲': Qianxiao, 
    '莫宁': Moning, 
    '琳奈': Linnai,
    '爱弥斯': Aimisi, 
    '守岸人': Shouanren, 
    '珂莱塔': Kelaita
}

In [6]:
# --- 3. 游戏引擎 ---
class FlyingChessGame:
    def __init__(self, selected_names=None, record_history=False):
        if selected_names is None: selected_names = list(AVAILABLE_TUANZIS.keys())
        self.tuanzis = [AVAILABLE_TUANZIS[name](name) for name in selected_names]
        self.boss = Tuanzi('Boss')
        self.turn_order = []
        self.board = {i: [] for i in range(1, TRACK_LENGTH + 1)}
        self.winner = None
        self.record_history = record_history
        self.history = []
        
        # 新增记录每轮所有人掷骰信息
        self.current_movement_rolls = {} 
        self.current_roll_info = {}
        self.round_logs = {} 
        
        self.round_num = 0
        self.turn_order = self.roll_turn_order(self.tuanzis)
        self.board[1] = self.turn_order[::-1]
        self.record_state("Round 0：游戏初始堆叠完毕（此顺位即为第一轮顺序）")
        self.round_num = 1

    def roll_turn_order(self, candidates):
        if not candidates: return []
        rolls = {t: t.roll() for t in candidates}
        groups = {}
        for t, r in rolls.items(): groups.setdefault(r, []).append(t)
        final = []
        for r in sorted(groups.keys(), reverse=True):
            if len(groups[r]) == 1: final.append(groups[r][0])
            else: final.extend(self.roll_turn_order(groups[r]))
        return final

    def get_rankings(self):
        def sort_key(t):
            if t.is_boss: return -999
            stack = self.board[t.pos]
            return (t.abs_dist, stack.index(t) if t in stack else -1)
        return sorted(self.tuanzis, key=sort_key, reverse=True)

    def move_stack(self, bottom_tuanzi, steps, is_backward=False):
        curr_pos = bottom_tuanzi.pos
        stack = self.board[curr_pos]
        idx = stack.index(bottom_tuanzi)
        moving_group = stack[idx:]
        
        self.board[curr_pos] = stack[:idx]
        for t in moving_group:
            if t.is_boss:
                t.abs_dist += -steps if is_backward else steps
            else:
                if is_backward: t.abs_dist -= steps
                else: t.abs_dist = min(TRACK_LENGTH, t.abs_dist + steps)
        
        new_pos = bottom_tuanzi.pos
        target_stack = self.board[new_pos]
        
        if any(t.is_boss for t in moving_group):
            self.board[new_pos] = moving_group + target_stack
        elif any(t.is_boss for t in target_stack):
            boss_idx = [t.is_boss for t in target_stack].index(True)
            self.board[new_pos] = target_stack[:boss_idx+1] + moving_group + target_stack[boss_idx+1:]
        else:
            self.board[new_pos] = target_stack + moving_group
            
        return moving_group, new_pos

    def resolve_traps_and_skills(self, moving_group, new_pos, trigger_tuanzi):
        trap = TRAPS.get(new_pos)
        if not trap: return "", 0

        if trap == 'BlackHole':
            stack = self.board[new_pos]
            boss = [t for t in stack if t.is_boss]
            others = [t for t in stack if not t.is_boss]
            random.shuffle(others)
            self.board[new_pos] = boss + others
            return "触发黑洞，发生乱序重排", 0
            
        elif trap in ['Propel', 'Block']:
            default_step = 1 if trap == 'Propel' else -1
            final_step = trigger_tuanzi.on_trap_trigger(self, trap, default_step)
            trap_name = "推进装置" if trap == 'Propel' else "阻遏装置"
            if trigger_tuanzi.is_boss: dir_str = "再次反向" if final_step > 0 else "被拉回"
            else: dir_str = "向前" if final_step > 0 else "向后"
            msg = f"落点触发{trap_name}，{dir_str}移动{abs(final_step)}格"
            
            is_backward = (final_step < 0)
            if trigger_tuanzi.is_boss: is_backward = not is_backward
            self.move_stack(trigger_tuanzi, abs(final_step), is_backward=is_backward)
            return msg, final_step

    def trigger_encounters(self):
        for t in self.tuanzis:
            t.check_encounter(self)
            if t.action_log:
                self.record_state(f"【被动生效】{t.name} {t.action_log}", t)
                t.action_log = ""

    def record_state(self, action_msg, actor=None):
        if not self.record_history: return
        board_copy = {k: [t.name for t in v] for k, v in self.board.items()}
        self.history.append({
            'round': self.round_num,
            'msg': action_msg,
            'board': board_copy,
            'ranks': [t.name for t in self.get_rankings()],
            'rolls': self.current_roll_info.copy() # 保存当轮掷骰信息供UI渲染
        })

    def check_winner(self):
        ranks = self.get_rankings()
        if ranks[0].abs_dist >= TRACK_LENGTH:
            self.winner = ranks[0]
            self.record_state(f"游戏结束！{self.winner.name} 抵达终点获胜！")
            return True
        return False

    def play_step(self):
        if self.winner: return True
        participants = self.tuanzis[:]

        if self.round_num > 1:
            if self.round_num >= 3:
                if self.round_num == 3 and not any(self.boss in v for v in self.board.values()):
                    self.board[1].insert(0, self.boss)
                    self.record_state("第3回合：Boss降临！")
                participants.append(self.boss)
            self.turn_order = self.roll_turn_order(participants)
            if self.record_history:
                self.record_state(f"重新掷骰排序，本轮顺序: {','.join([t.name for t in self.turn_order])}")
        else:
            if self.record_history:
                self.record_state(f"第一轮开始，初始顺序: {','.join([t.name for t in self.turn_order])}")

        # 记录本轮移动掷骰结果
        self.current_movement_rolls = {t: t.movement_roll() for t in participants}
        self.current_roll_info = {t.name: self.current_movement_rolls[t] for t in self.turn_order}
        self.round_logs[self.round_num] = self.current_roll_info.copy()

        for t in self.tuanzis: t.sigelika_debuff = 0
        for t in self.tuanzis: 
            t.on_round_start(self)
            if t.action_log: 
                self.record_state(f"{t.name} 发动技能: {t.action_log}", t)
                t.action_log = ""

        for current_t in self.turn_order:
            if self.winner: break
            
            roll = self.current_movement_rolls[current_t]
            
            if current_t.is_boss:
                base_steps = roll
                m_group, n_pos = self.move_stack(current_t, base_steps, is_backward=True)
                self.trigger_encounters()
                self.record_state(f"Boss掷骰{roll}，反向移动{base_steps}格", current_t)
            else:
                base_steps = current_t.modify_steps(self, roll)
                if base_steps <= 0:
                    msg = f"{current_t.name} 掷骰{roll}"
                    if current_t.action_log:
                        msg += f" [{current_t.action_log}]"
                        current_t.action_log = ""
                    self.record_state(msg, current_t)
                else:
                    m_group, n_pos = self.move_stack(current_t, base_steps)
                    self.trigger_encounters()
                    msg = f"{current_t.name} 掷骰{roll}"
                    if current_t.action_log:
                        msg += f" [{current_t.action_log}]"
                        current_t.action_log = ""
                    msg += f"，向前移动{base_steps}格"
                    self.record_state(msg, current_t)

            if self.check_winner(): return True

            if current_t.is_boss or base_steps > 0:
                trap_msg, trap_steps = self.resolve_traps_and_skills(m_group, n_pos, current_t)
                if trap_msg:
                    self.trigger_encounters()
                    self.record_state(f"{current_t.name} {trap_msg}", current_t)
                    if self.check_winner(): return True

            if not current_t.is_boss:
                current_t.on_turn_end(self)
                if current_t.action_log: 
                    self.record_state(f"{current_t.name}: {current_t.action_log}", current_t)
                    current_t.action_log = ""

        if self.round_num >= 3:
            last_place = self.get_rankings()[-1]
            if self.boss.pos < last_place.pos:
                 b_stack = self.board[self.boss.pos]
                 if self.boss in b_stack: b_stack.remove(self.boss)
                 self.boss.abs_dist = 0
                 self.board[1].insert(0, self.boss)
                 self.record_state(f"Boss传送回起点！")

        self.round_num += 1
        return False

In [31]:
# --- 4. 可视化 ---
def visualize_game_jupyter(game=None, selected_names=None):
    if game is None:
        game = FlyingChessGame(selected_names=selected_names, record_history=True)
        while not game.play_step(): pass

    fig, ax = plt.subplots(figsize=(12, 12))
    fig.patch.set_facecolor('#F4F4F4')
    ax.set_facecolor('#F4F4F4')
    plt.close(fig) 
    
    angles = [math.pi/2 - i*(2*math.pi/TRACK_LENGTH) for i in range(TRACK_LENGTH)]
    R = 10 

    def update(frame):
        ax.clear()
        ax.set_xlim(-R-14, R+16)
        ax.set_ylim(-R-14, R+16)
        ax.axis('off')
        
        for i in range(1, TRACK_LENGTH + 1):
            angle = angles[i-1]
            x, y = R * math.cos(angle), R * math.sin(angle)
            color = '#E0E0E0'
            if i in TRAPS:
                if TRAPS[i] == 'Propel': color = '#B3E5FC'
                elif TRAPS[i] == 'Block': color = '#FFCCBC'
                elif TRAPS[i] == 'BlackHole': color = '#9E9E9E'
            if i == 1: color = '#FFF59D'
            ax.add_patch(plt.Circle((x, y), 0.6, color=color, ec='#757575', zorder=1))
            ax.text(x, y, str(i), ha='center', va='center', fontsize=8, color='#424242', zorder=2)
            
        state = game.history[frame]
        for pos, stack in state['board'].items():
            if not stack: continue
            angle = angles[pos-1]
            for stack_idx, t_name in enumerate(stack):
                offset = 1.0 + stack_idx * 0.7
                x = (R + offset) * math.cos(angle)
                y = (R + offset) * math.sin(angle)
                ax.add_patch(plt.Circle((x, y), 0.45, color=COLORS.get(t_name, 'black'), ec='white', lw=1.2, zorder=3))
                ax.text(x, y, t_name[0], ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)

        # 左上角排名
        ax.text(-R-13, R+7, f"Round: {state['round']}", fontsize=14, fontweight='bold', color='#333333')
        ax.text(-R-13, R+5, "当前排名:", fontsize=12, color='#333333')
        for i, r_name in enumerate(state['ranks']):
            ax.text(-R-13, R+3.5 - i*1.0, f"{i+1}. {r_name}", color=COLORS.get(r_name, 'black'), fontsize=11, fontweight='bold')
            
        # 右上角：当轮移动预掷骰面板
        rolls = state.get('rolls', {})
        if rolls:
            ax.text(R+10, R+9, "本轮行动顺序与掷骰结果", fontsize=13, fontweight='bold', color='#333333')
            for i, (name, roll) in enumerate(rolls.items()):
                ax.text(R+10, R+7.5 - i*1.0, f"{name}: {roll}", color=COLORS.get(name, 'black'), fontsize=12, fontweight='bold')
            
        # 底部消息提示框
        ax.text(0, -R-9, state['msg'], ha='center', fontsize=12, color='#212121', 
                bbox=dict(facecolor='white', edgecolor='#BDBDBD', boxstyle='round,pad=0.5', alpha=0.9))

    ani = animation.FuncAnimation(fig, update, frames=len(game.history), interval=650, repeat=False)
    return HTML(ani.to_jshtml())


def simulate_win_rates(trials=1000, selected_names=None):
    if selected_names is None: selected_names = list(AVAILABLE_TUANZIS.keys())
    print(f"开始蒙特卡洛模拟，参赛者: {', '.join(selected_names)}，局数: {trials}...")
    win_counts = {name: 0 for name in selected_names}
    
    for _ in range(trials):
        game = FlyingChessGame(selected_names=selected_names, record_history=False)
        while not game.play_step(): pass
        win_counts[game.winner.name] += 1

    rates = [win_counts[n] / trials * 100 for n in selected_names]
    
    print("\n--- 模拟胜率结果 ---")
    for name, rate in zip(selected_names, rates): print(f"{name:8s} | {rate:5.2f}%")
        
    fig, ax = plt.subplots(figsize=(max(8, len(selected_names)*1.2), 6))
    fig.patch.set_facecolor('#F8F9FA')
    ax.set_facecolor('#F8F9FA')
    
    bars = ax.bar(selected_names, rates, color=[COLORS.get(n, 'black') for n in selected_names], edgecolor='#2C2C2C', linewidth=1.5, width=0.6)
    
    ax.yaxis.grid(True, linestyle='--', color='#D3D3D3', alpha=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_title(f'Win Rate Distribution ($N={trials}$)', fontsize=16, fontweight='bold', pad=15)
    ax.set_ylabel('Win Rate (%)', fontsize=12, fontweight='bold')
    ax.set_ylim(0, max(rates) + 10)
    
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 0.8, f"{yval:.1f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')
        
    plt.show()

In [ ]:
from IPython.display import clear_output

def display_interactive_dashboard():
    checkboxes = {
        name: widgets.Checkbox(value=True, description=name, layout={'width': 'max-content'})
        for name in AVAILABLE_TUANZIS.keys()
    }
    
    cb_container = widgets.VBox(list(checkboxes.values()))
    
    trials_input = widgets.IntText(value=5000, description='模拟次数:', layout={'width': '200px'})
    btn_anim = widgets.Button(description="▶ 观看一局动画", button_style='info', icon='play')
    btn_sim = widgets.Button(description="📊 统计胜率分布", button_style='success', icon='bar-chart')
    output_area = widgets.Output()

    def get_selected_names():
        return [name for name, cb in checkboxes.items() if cb.value]

    def on_anim_clicked(b):
        selected = get_selected_names()
        with output_area:
            clear_output(wait=True)
            if not selected:
                print("⚠️ 请至少选择一名团子参赛！")
            else:
                game = FlyingChessGame(selected_names=selected, record_history=True)
                while not game.play_step(): pass
                
                # log_html = """
                # <div style='max-height: 180px; overflow-y: auto; border: 1px solid #ccc; padding: 10px; 
                #             background-color: #f8f9fa; border-radius: 5px; margin-bottom: 10px;'>
                #     <h4 style='margin-top:0;'>🎲 本局游戏每轮掷骰与行动顺位记录</h4>
                # """
                # for r, rolls in game.round_logs.items():
                #     # 字典已被排序，按行动顺序输出，并加上序号
                #     roll_strs = [f"<b>{i+1}. </b><span style='color:{COLORS.get(name, 'black')}'><b>{name}</b>:{roll}</span>" 
                #                  for i, (name, roll) in enumerate(rolls.items())]
                #     log_html += f"<div style='margin-bottom: 4px; font-size: 13px;'><b>Round {r}</b>: " + " | ".join(roll_strs) + "</div>"
                # log_html += "</div>"
                
                # display(widgets.HTML(log_html))
                display(visualize_game_jupyter(game))

    def on_sim_clicked(b):
        selected = get_selected_names()
        trials = trials_input.value
        with output_area:
            clear_output(wait=True)
            if not selected:
                print("⚠️ 请至少选择一名团子参赛！")
            elif trials <= 0:
                print("⚠️ 模拟次数必须大于0！")
            else:
                simulate_win_rates(trials, selected)

    btn_anim.on_click(on_anim_clicked)
    btn_sim.on_click(on_sim_clicked)
    
    ui = widgets.VBox([
        widgets.HTML("<h3>🎯 团子飞行棋 - 模拟控制台</h3><hr>"),
        widgets.HTML("<b>步骤1：选择参赛阵容</b>"),
        cb_container,
        widgets.HTML("<br><b>步骤2：选择执行模式</b>"),
        widgets.HBox([trials_input, btn_sim, btn_anim]),
        widgets.HTML("<hr>"),
        output_area
    ])
    
    display(ui)

display_interactive_dashboard()

In [9]:
# 模拟胜率
# trials = int(input("请输入要模拟的次数: ") )
# simulate_win_rates(trials)

In [10]:
# 生成单局动画，可以直接在输出框通过播放组件控制进度条、播放/暂停
# display(visualize_game_jupyter())